In [38]:
import numpy as np
import random as rd
from itertools import product as iproduct
from collections import Counter

In [39]:
# ============================================================
# PARAMÈTRES
# ============================================================

MARKER_PENALTY_H2 = 0.15
N_SIMULATIONS = 2
MAX_TURNS = 1000

COLUMNS = range(2, 13)

Ns = {
    2: 3, 3: 5, 4: 7, 5: 9, 6: 11,
    7: 13,
    8: 11, 9: 9, 10: 7, 11: 5, 12: 3
}

In [40]:
# ============================================================
# PROBABILITÉS p_c
# ============================================================

def _compute_prob_col(target):
    count = 0
    for roll in iproduct(range(1, 7), repeat=4):
        d1, d2, d3, d4 = roll
        sums = {
            d1+d2, d3+d4,
            d1+d3, d2+d4,
            d1+d4, d2+d3
        }
        if target in sums:
            count += 1
    return count / 1296

PROB_COL = {c: _compute_prob_col(c) for c in COLUMNS}
ALL_ROLLS = list(iproduct(range(1, 7), repeat=4))

In [41]:
# ============================================================
# UTILITAIRES
# ============================================================

def get_pairings(roll):
    d1, d2, d3, d4 = roll

    pairings = [
        tuple(sorted((d1+d2, d3+d4))),
        tuple(sorted((d1+d3, d2+d4))),
        tuple(sorted((d1+d4, d2+d3))),
    ]

    seen = set()
    unique = []
    for p in pairings:
        if p not in seen:
            seen.add(p)
            unique.append(p)

    return unique


def action_is_legal(action, progress, temp_progress):
    new_cols = set()

    for s in action:
        if progress[s] + temp_progress.get(s, 0) >= Ns[s]:
            return False
        if s not in temp_progress:
            new_cols.add(s)

    return len(temp_progress) + len(new_cols) <= 3


def apply_action(action, progress, temp_progress):
    new_temp = temp_progress.copy()

    for s in action:
        if progress[s] + new_temp.get(s, 0) < Ns[s]:
            new_temp[s] = new_temp.get(s, 0) + 1

    return new_temp


def bank_progress(progress, temp_progress):
    new_progress = progress.copy()
    for s, inc in temp_progress.items():
        new_progress[s] = min(new_progress[s] + inc, Ns[s])
    return new_progress


def nb_won_columns(progress):
    return sum(progress[s] >= Ns[s] for s in COLUMNS)

In [42]:
# ============================================================
# H1 — MINIMISER OUVERTURES
# ============================================================

def score_h1_no_open(action, progress, temp_progress):
    score = 0
    for s in action:
        score += 6 - abs(7 - s)
        remaining = Ns[s] - progress[s] - temp_progress.get(s, 0)
        if remaining <= 2:
            score += 3
    return score


def choose_action_h1(progress, temp_progress, legal_actions):
    no_open = [a for a in legal_actions if all(s in temp_progress for s in a)]

    if no_open:
        return max(no_open, key=lambda a: score_h1_no_open(a, progress, temp_progress))

    def open_value(a):
        val = 0
        for s in set(a):
            if s not in temp_progress:
                remaining = max(Ns[s] - progress[s] - temp_progress.get(s, 0), 1)
                val += PROB_COL[s] / remaining
        return val

    return max(legal_actions, key=open_value)


def heuristic_h1(progress, temp_progress, legal_actions, turn_roll_count):
    return choose_action_h1(progress, temp_progress, legal_actions), (turn_roll_count >= 3)

In [43]:
# ============================================================
# H2 — p_c / restants
# ============================================================

def col_value(s, progress, temp_progress):
    current = progress[s] + temp_progress.get(s, 0)
    remaining = max(Ns[s] - current, 1)
    return PROB_COL[s] / remaining


def score_h2(action, progress, temp_progress):
    score = 0
    for s, mult in Counter(action).items():
        score += mult * col_value(s, progress, temp_progress)
        if s not in temp_progress:
            score -= MARKER_PENALTY_H2
    return score


def heuristic_h2(progress, temp_progress, legal_actions, turn_roll_count):
    action = max(legal_actions, key=lambda a: score_h2(a, progress, temp_progress))
    return action, (turn_roll_count >= 3)

In [44]:
# ============================================================
# H3 — MINIMISER BUST
# ============================================================

def bust_prob_after_action(action, progress, temp_progress):
    hyp = temp_progress.copy()

    for s in action:
        if progress[s] + hyp.get(s, 0) < Ns[s]:
            hyp[s] = hyp.get(s, 0) + 1

    busts = 0

    for roll in ALL_ROLLS:
        pairings = get_pairings(roll)
        if not any(action_is_legal(p, progress, hyp) for p in pairings):
            busts += 1

    return busts / 1296


def heuristic_h3(progress, temp_progress, legal_actions, turn_roll_count):
    probs = [bust_prob_after_action(a, progress, temp_progress) for a in legal_actions]
    min_p = min(probs)

    candidates = [a for a, p in zip(legal_actions, probs) if p == min_p]

    action = max(candidates, key=lambda a: score_h2(a, progress, temp_progress))
    return action, (turn_roll_count >= 3)

In [45]:
# ============================================================
# SIMULATION 1 PARTIE
# ============================================================

def simulate_game(heuristic_fn):

    progress = {s: 0 for s in COLUMNS}
    n_turns = 0

    while nb_won_columns(progress) < 3:

        temp_progress = {}
        turn_roll_count = 0

        while True:

            roll = tuple(np.random.randint(1, 7, 4))
            turn_roll_count += 1

            pairings = get_pairings(roll)

            legal = [a for a in pairings if action_is_legal(a, progress, temp_progress)]

            if not legal:
                break  # bust

            action, stop = heuristic_fn(progress, temp_progress, legal, turn_roll_count)
            temp_progress = apply_action(action, progress, temp_progress)

            if stop:
                progress = bank_progress(progress, temp_progress)
                break

        n_turns += 1

        if n_turns > MAX_TURNS:
            return MAX_TURNS

    return n_turns

# ============================================================
# BENCHMARK
# ============================================================

def benchmark():

    heuristics = {
        "H1": heuristic_h1,
        "H2": heuristic_h2,
        "H3": heuristic_h3,
    }

    results = {}

    for name, h in heuristics.items():
        scores = []

        for _ in range(N_SIMULATIONS):
            score = simulate_game(h)
            scores.append(score)

        results[name] = {
            "mean": np.mean(scores),
            "std": np.std(scores),
            "min": np.min(scores),
            "max": np.max(scores),
        }

    return results

In [46]:
# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    results = benchmark()

    print("\n=== RÉSULTATS ===\n")
    for h, stats in results.items():
        print(f"{h}:")
        print(f"  mean turns : {stats['mean']:.2f}")
        print(f"  std       : {stats['std']:.2f}")
        print(f"  min/max   : {stats['min']} / {stats['max']}")
        print()


=== RÉSULTATS ===

H1:
  mean turns : 52.50
  std       : 24.50
  min/max   : 28 / 77

H2:
  mean turns : 37.50
  std       : 14.50
  min/max   : 23 / 52

H3:
  mean turns : 40.00
  std       : 6.00
  min/max   : 34 / 46

